In [1]:
# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # maxOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
    
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

SB_DEEP_GREEN = '#1E3932'
SB_GREEN      = '#006241'
SB_LIGHT_GREEN = '#D4E9E2'
SB_GOLD       = '#CBA258'
SB_GREY       = '#A2AAAD'
SB_BLACK      = '#27251F'

plt.rcParams.update({
    'font.family': 'Malgun Gothic',
    'axes.unicode_minus': False,
    'text.color': SB_BLACK,
    'axes.labelcolor': SB_BLACK,
    'xtick.color': SB_BLACK,
    'ytick.color': SB_BLACK,
    'axes.spines.top': False,    
    'axes.spines.right': False, 
    'patch.edgecolor': 'none'    
})


sns.set_palette([SB_GREEN, SB_GOLD, SB_DEEP_GREEN, SB_LIGHT_GREEN, SB_GREY])

In [3]:
merge_df = pd.read_csv("../../Data/merged_df_260325.csv")

In [4]:
trans_df = pd.read_csv("../../Data/transactions_260325.csv")

In [27]:
import pandas as pd
import ast

# same-family multi를 첫 번째 label에 귀속시켜 성과가 과대 반영 <보강 ㅠ
def get_correct_label(row):
    # 1. 아예 종류가 다른 오퍼들이 섞인 경우 (원래부터 multi)
    if row['tx_offer_type'] == 'multi': 
        return '다중오퍼'
    
    # 2. bogo나 discount인데, 안에서 여러 개 깬 경우 (same-family multi)
    elif row['tx_offer_type'] in ['bogo', 'discount']:
        labels = ast.literal_eval(row['reward_offer_label_list'])
        
        
        if len(labels) > 1:
            return '다중오퍼'  # 2개 이상 깼으면 무조건 다중오퍼로 분리
        elif len(labels) == 1:
            return labels[0]   # 오퍼1개당 컴플리트 1개만 깼으면 그 오퍼 이름 부여
            
    return '일반결제'

# trans_df에 'final_label' 컬럼 장착
trans_df['final_label'] = trans_df.apply(get_correct_label, axis=1)

# 유지보수성 보강 - trans_df 라벨링

# 1. trans_df에서 '다중오퍼'로 판명된 결제들의 사람/시간 정보 추출
multi_txn_info = trans_df[trans_df['final_label'] == '다중오퍼'][['person', 'time']].drop_duplicates()
multi_txn_info['is_multi_txn'] = True

# 2. merge_df의 완료 이벤트와 엮어서 '다중오퍼 경로' 키(key) 찾아내기
completions = merge_df[merge_df['event'] == 'completed']
multi_path = pd.merge(completions, multi_txn_info, on=['person', 'time'], how='inner')
multi_path_keys = multi_path[['person', 'offer_id', 'receive_seq']].drop_duplicates()
multi_path_keys['is_multi_path'] = True

# 3. merge_df 전체(인포메이셔널 제외)에 'final_label' 장착
master_df = merge_df[~merge_df['offer_label'].str.contains('informational', na=False)].copy()
master_df = pd.merge(master_df, multi_path_keys, on=['person', 'offer_id', 'receive_seq'], how='left')

# 다중오퍼 경로면 '다중오퍼', 아니면 원래 분류 붙이기 
master_df['final_label'] = master_df.apply(
    lambda x: '다중오퍼' if x['is_multi_path'] == True else x['offer_label'], axis=1
)